In [0]:
from pyspark.sql import functions as F

#Reading from bronze table

In [0]:
df = (
    spark.read.table("albion_project.bronze.prices")
)
display(df)

#Cleaning null prices

In [0]:
silver_df = (
    df
    .withColumn(
        "has_sell_offer",
        (F.col("sell_price_min") > 0) & (F.col("sell_price_max") > 0)
    )
    .withColumn(
        "has_buy_offer",
        (F.col("buy_price_min") > 0) & (F.col("buy_price_max") > 0)
    )
)

silver_df=(
    silver_df.where(
        F.col("has_sell_offer") | F.col("has_buy_offer")
    )
)

#Transforming dates to date format

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "sell_price_min_date",
        F.to_timestamp(
            F.col("sell_price_min_date"),
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )
    .withColumn(
        "sell_price_max_date",
        F.to_timestamp(
            F.col("sell_price_max_date"),
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )
    .withColumn(
        "buy_price_min_date",
        F.to_timestamp(
            F.col("buy_price_min_date"),
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )
    .withColumn(
        "buy_price_max_date",
        F.to_timestamp(
            F.col("buy_price_max_date"),
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )
)
silver_df.printSchema()
display(silver_df)

#Adding "has_both_offers" column

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "has_both_offers",
        F.col("has_sell_offer") & F.col("has_buy_offer")
    )
)
display(silver_df)

#Converting 0 prices to nulls

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "sell_price_min",
        F.when(
            ~F.col("has_sell_offer"),
            F.lit(None)
        ).otherwise(
            F.col("sell_price_min")
        )
    )
    .withColumn(
        "sell_price_max",
        F.when(
            ~F.col("has_sell_offer"),
            F.lit(None)
        ).otherwise(
            F.col("sell_price_max")
        )
    )
    .withColumn(
        "sell_price_min_date",
        F.when(
            ~F.col("has_sell_offer"),
            F.lit(None)
        ).otherwise(
            F.col("sell_price_min_date")
        )
    )
    .withColumn(
        "sell_price_max_date",
        F.when(
            ~F.col("has_sell_offer"),
            F.lit(None)
        ).otherwise(
            F.col("sell_price_max_date")
        )
    )
    .withColumn(
        "buy_price_max",
        F.when(
            ~F.col("has_buy_offer"),
            F.lit(None)
        ).otherwise(
            F.col("buy_price_max")
        )
    )
    .withColumn(
        "buy_price_min",
        F.when(
            ~F.col("has_buy_offer"),
            F.lit(None)
        ).otherwise(
            F.col("buy_price_min")
        )
    )
    .withColumn(
        "buy_price_max_date",
        F.when(
            ~F.col("has_buy_offer"),
            F.lit(None)
        ).otherwise(
            F.col("buy_price_max_date")
        )
    )
    .withColumn(
        "buy_price_min_date",
        F.when(
            ~F.col("has_buy_offer"),
            F.lit(None)
        ).otherwise(
            F.col("buy_price_min_date")
        )
    )
)
silver_df.printSchema()

#Adding processed_at column

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "processed_at",
        F.current_timestamp()
    )
)
display(silver_df)
silver_df.printSchema()

#Adding enchant column

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "enchant",
        F.when(
            F.col("item_id").rlike(r"@([0-4])$"),
            F.regexp_extract(
                F.col("item_id"),
                r"@([0-4])$",
                1
            ).cast("int")
        ).otherwise(
            F.lit(0)
        )
    )
)

#Adding base_item_id

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "base_item_id",
        F.regexp_replace(
            F.col("item_id"),
            r"@([0-4])$",
            ""
        )
    )
)

#Writing to silver table

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.silver.prices")
)

In [0]:
display(silver_df.count())

#